# Jupyter notebook used to plot the data for the TBird Nature Manuscript
## The images contained here are those found in the Nature Manuscript, as are calculations for neutron production rates and the stability criteria

In [ ]:
%matplotlib inline
import numpy as np
import scipy
import statistics
import matplotlib as mpl
from matplotlib import gridspec
import matplotlib.ticker as ticker
from scipy.optimize import curve_fit
from matplotlib.ticker import (MultipleLocator, AutoMinorLocator, FormatStrFormatter)
from scipy import interpolate
import matplotlib.patches as mpatches
import pandas as pd 
from numpy import *
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.signal import find_peaks
import matplotlib.ticker as plticker
# import probfit
from scipy import special
import time
import datetime

mpl.rc('font', family='Arial')

### Reading and writing data to dataframes for easier access

In [ ]:
def readNeutronData(filename):
    tmp = pd.read_csv(filename, sep=',', header = None, skiprows=1)
    tmp.columns = ['binMid', 'binTime(s)', 'Neutrons(cps)','delta-neutrons', 'BackgroundGamma(cps)', 'delta-gamma']
    tmp['binTime(m)'] = tmp['binTime(s)']/60
    return tmp

In [ ]:
# Data for target A5 (target 1 in manuscript)
data_folder = "C:/Users/oliver.horner/Desktop/Nature paper raw data/Output/"
id419 = readNeutronData(data_folder + 'ID-419/ID-419_data_300s_bin.csv')
id423 = readNeutronData(data_folder + 'ID-423/ID-423_data_300s_bin.csv')

# Data for target A6 (target 2 in manuscript)
id421 = readNeutronData(data_folder + 'ID-421/ID-421_data_300s_bin.csv')
id418 = readNeutronData(data_folder + 'ID-418/ID-418_data_300s_bin.csv')

# Data for target A7 (target 3 in manuscript)
id415 = readNeutronData(data_folder + 'ID-415/ID-415_data_300s_bin.csv')
id422 = readNeutronData(data_folder + 'ID-422/ID-422_data_300s_bin.csv')

id379 = readNeutronData(data_folder + 'ID-379/ID-379_data_300s_bin.csv')


In [ ]:
## Calculation of average values

def avgMath(df, tlow, thigh):
    tmp = df[(df['binTime(s)'] > tlow) & (df['binTime(s)'] < thigh)]
    avg = tmp['Neutrons(cps)'].mean()
    stdDerr =statistics.stdev(tmp['Neutrons(cps)'])/sqrt(len(tmp['Neutrons(cps)']))
    return avg, stdDerr

def percentError(df1, df2, tlow, thigh):
    tmp1 = df1[(df1['binTime(s)'] > tlow) & (df1['binTime(s)'] < thigh)]
    avg1 = tmp1['Neutrons(cps)'].mean()
    stdDerr1 = statistics.stdev(tmp1['Neutrons(cps)'])/sqrt(len(tmp1['Neutrons(cps)']))

    tmp2 = df2[(df2['binTime(s)'] > tlow) & (df2['binTime(s)'] < thigh)]
    avg2 = tmp2['Neutrons(cps)'].mean()
    stdDerr2 = statistics.stdev(tmp2['Neutrons(cps)'])/sqrt(len(tmp2['Neutrons(cps)']))

    diff = ((avg2 - avg1)/avg1)*100
    err = (sqrt((stdDerr2/avg2)**2 + 2*((stdDerr1/avg1)**2))*diff)
    
    return diff, err

In [ ]:
# t_lo = 95*60  # was 4600s
# t_hi = 125*60  # was 7500s (same value)
t_lo = 4600
t_hi = 7500

print('----------Target A5-----------------')
avg419, err419 = avgMath(id419, t_lo, t_hi)
print("Average counts (beam) = %.2f(%.2f)" % (avg419, err419))

avg423, err423 = avgMath(id423, t_lo, t_hi)
print("Average counts (beam+e-cell) = %.2f(%.2f)" % (avg423, err423))

diffA5, errA5 = percentError(id419, id423, t_lo, t_hi)
print("Percent difference(A5) = %.2f(%.2f)" % (diffA5, errA5))

print('----------Target A6------------------')
avg421, err421 = avgMath(id421, t_lo, t_hi)
print("Average counts (beam) = %.2f(%.2f)" % (avg421, err421))

avg418, err418 = avgMath(id418, t_lo, t_hi)
print("Average counts (beam+e-cell) = %.2f(%.2f)" % (avg418, err418))

diffA6, errA6 = percentError(id421, id418, t_lo, t_hi)
print("Percent difference(A6) = %.2f(%.2f)" % (diffA6, errA6))

print('----------Target A7------------------')
avg415, err415 = avgMath(id415, t_lo, t_hi)
print("Average counts (beam) = %.2f(%.2f)" % (avg415, err415))

avg422, err422 = avgMath(id422, t_lo, t_hi)
print("Average counts (beam+e-cell) = %.2f(%.2f)" % (avg422, err422))

diffA7, errA7 = percentError(id415, id422, t_lo, t_hi)
print("Percent difference(A7) = %.2f(%.2f)" % (diffA7, errA7))

print('----------Background-----------------')
avg379, err379 = avgMath(id379, t_lo, t_hi)
print("Average counts = %.2f(%.2f)" % (avg379, err379))


In [ ]:
# Calculation for the average difference
runAvgs = [diffA5, diffA6, diffA7]
errorValues = [errA5, errA6, errA7]

totalAvg = sum(runAvgs)/len(runAvgs)
totalError = statistics.stdev(runAvgs)/sqrt(len(runAvgs))

print("Percent difference(total) = %.2f(%.2f)" % (totalAvg, totalError))

## The official plotting mode for the iconic plot in the manuscript (Figure 3)

In [ ]:
fig, axs = plt.subplots(1, 3, figsize = (20,7))
fig.tight_layout()

axs[0].errorbar(id419['binTime(m)'], id419['Neutrons(cps)'], id419['delta-neutrons'], ls ='', marker = 'o', ms = 10, capsize = 0, label = 'Beam loaded', color ='#424242FF' )
axs[0].errorbar(id423['binTime(m)'], id423['Neutrons(cps)'], id423['delta-neutrons'], ls ='', marker = 's', ms = 10, capsize = 0, label = 'Electrochemically loaded', color ='#941100FF' )
# axs[0].legend(fontsize = 14, loc = 'lower right')
# axs[0].set_title('Target A1', fontsize = 32)
axs[0].tick_params(axis="y", labelsize = 30)
axs[0].set_xlim(-4,125)
axs[0].tick_params(axis="x", labelsize = 30)
axs[0].set_ylim(-5,190)
axs[0].xaxis.set_minor_locator(AutoMinorLocator(3))
axs[0].yaxis.set_minor_locator(AutoMinorLocator(3))
axs[0].set_ylabel('Neutron production rate (1/s)', fontsize=30)
axs[0].axhline(132, color = 'black', ls = "--", alpha = 0.7)
axs[0].axhline(155, color = 'black', ls = "--", alpha = 0.7)
axs[0].axvline(65, color = 'black', ls = "-", alpha = 0.7)
# axs[0].annotate('E-cell start \n15% increase',(4000,115), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[0].annotate('15% increase',(6,141), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
loc = plticker.MultipleLocator(base=30) # this locator puts ticks at regular intervals
axs[0].xaxis.set_major_locator(loc)
axs[0].axvspan(5,65, alpha=0.1, color='#4574A2FF')
axs[0].axvspan(-4,5, alpha=0.1, color='gray')
axs[0].axhspan(132,155, alpha=0.1, color='gray')

axs[0].annotate('Phase I',(20,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs[0].annotate('Phase II',(82,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )


axs[1].errorbar(id421['binTime(m)'], id421['Neutrons(cps)'], id421['delta-neutrons'], ls ='', marker = 'o', ms = 10, capsize = 0, label = 'Beam loaded', color = '#424242FF')
axs[1].errorbar(id418['binTime(m)'], id418['Neutrons(cps)'], id418['delta-neutrons'], ls ='', marker = 's', ms = 10, capsize = 0, label = 'Electrochemically loaded', color = '#941100FF')
axs[1].axhline(140, color = 'black', ls = "--", alpha = 0.7)
axs[1].axhline(156, color = 'black', ls = "--", alpha = 0.7)
# axs[1].set_title('Target A2', fontsize = 30)
axs[1].xaxis.set_minor_locator(AutoMinorLocator(3))
axs[1].yaxis.set_minor_locator(AutoMinorLocator(3))
# axs[1].tick_params(axis="y")
axs[1].set_ylim(-5,190)
axs[1].set_xlim(-4,125)
# axs[1].legend(fontsize = 14, loc = 'lower right')
axs[1].set_xlabel('Time (min)', fontsize = 32)
# axs[1].annotate('E-cell start \n11% increase',(4300,120), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[1].annotate('11% increase',(6,144), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24)
axs[1].axvline(70, color = 'black', ls = "-", alpha = 0.7)
# axs[1] = axs[1].twinx()
axs[1].tick_params(axis="y")
axs[1].tick_params(axis="x", labelsize = 30)
loc = plticker.MultipleLocator(base=30) 
axs[1].xaxis.set_major_locator(loc)
axs[1].axvspan(5,70, alpha=0.1, color='#4574A2FF')
axs[1].axvspan(-4,5, alpha=0.1, color='gray')
axs[1].axhspan(140,156, alpha=0.1, color='gray')

axs[1].annotate('Phase I',(20,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs[1].annotate('Phase II',(81,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )


axs[2].errorbar(id415['binTime(m)'], id415['Neutrons(cps)'], id415['delta-neutrons'], ls ='', marker = 'o', ms = 10, capsize = 0, label = 'Beam loaded', color = '#424242FF')
axs[2].errorbar(id422['binTime(m)'], id422['Neutrons(cps)'], id422['delta-neutrons'], ls ='', marker = 's', ms = 10, capsize = 0, label = 'Electrochemically loaded', color = '#941100FF')
# axs[2].legend(fontsize = 14, loc = 'lower right')
# axs[2].set_title('Target A3', fontsize = 30)
axs[2].set_ylim(-5,190)
axs[2].xaxis.set_minor_locator(AutoMinorLocator(3))
axs[2].yaxis.set_minor_locator(AutoMinorLocator(3))
axs[2].set_xlim(-4,125)
axs[2].tick_params(axis="x", labelsize = 30)
axs[2].axhline(135, color = 'black', ls = "--", alpha = 0.7)
axs[2].axhline(162, color = 'black', ls = "--", alpha = 0.7)
axs[2].axvline(66, color = 'black', ls = "-", alpha = 0.7)
# axs[2].annotate('E-cell start \n18% increase',(4060,120), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
axs[2].annotate('18% increase',(6,145), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
loc = plticker.MultipleLocator(base=30)
axs[2].xaxis.set_major_locator(loc)
axs[2].axvspan(5,66, alpha=0.1, color='#4574A2FF')
axs[2].axvspan(-4,5, alpha=0.1, color='gray')
axs[2].axhspan(135,162, alpha=0.1, color='gray')

axs[2].annotate('Phase I',(20,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs[2].annotate('Phase II',(82,172), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )


# Hide x labels and tick labels for top plots and y ticks for right plots.
for ax in axs.flat:
    ax.label_outer()

plt.savefig("Fig3-IconicPlot-Updated.png", format="png", bbox_inches="tight")


-------------
# On/Off data beam loading plotting (SI Figure 7 - Proof of fusion in the lattice)

In [ ]:
id478 = readNeutronData(data_folder + 'ID-478/ID-478_data_300s_bin.csv')
id432 = readNeutronData(data_folder + 'ID-432/ID-432_data_300s_bin.csv')
id478.head()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
# fig.tight_layout()

axs.errorbar(id432['binTime(m)'], id432['Neutrons(cps)'], id432['delta-neutrons'], ls ='', marker = 'o', ms = 10, capsize = 0, color ='#424242FF' )
axs.tick_params(axis="y", labelsize = 24)
# axs.set_xlim(-4,125)
axs.tick_params(axis="x", labelsize = 24)
axs.set_ylim(-5,220)
axs.xaxis.set_minor_locator(AutoMinorLocator(3))
axs.yaxis.set_minor_locator(AutoMinorLocator(3))
axs.set_ylabel('Neutron production rate (1/s)', fontsize=24)
loc = plticker.MultipleLocator(base=15) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)
axs.axvspan(5,15, alpha=0.1, color='#4574A2FF')
axs.annotate('on',(8.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.axvspan(20,30, alpha=0.1, color='#4574A2FF')
axs.annotate('on',(23.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.axvspan(35,45, alpha=0.1, color='#4574A2FF')
axs.annotate('on',(38.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs.axvspan(-4,5, alpha=0.1, color='gray')

axs.annotate('off',(31, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.annotate('off',(16, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.annotate('off',(31, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.annotate('off',(46.5, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )


axs.set_xlabel('Time (min)', fontsize = 24)
plt.savefig("SI-Fig12-ThursterPowerCycle.pdf", format="pdf", bbox_inches="tight")
plt.show()

# for ax in axs.flat:
    # ax.label_outer()





-------------
# Effect of H2O use on neutron production rate

In [ ]:
tb1 = readNeutronData(data_folder + 'TB-1/TB-1_data_180s_bin.csv')
tb1.head()

In [ ]:
fig, axs = plt.subplots(1, 1, figsize=(7, 7), dpi=600)
# fig.tight_layout()

axs.errorbar(tb1['binTime(m)'], tb1['Neutrons(cps)'], tb1['delta-neutrons'], ls='', marker='o', ms=10, capsize=0, color='#424242FF')

axs.set_xlim(-5, 155)
axs.set_ylim(-5, 140)

axs.tick_params(labelsize=24)
axs.xaxis.set_major_locator(mpl.ticker.MultipleLocator(20))

axs.set_xlabel('Time (min)', fontsize = 24)
axs.set_ylabel('Neutron production rate (1/s)', fontsize=24)

axs.axvspan(-5, 12, alpha=0.1, color='grey')
axs.axvspan(12, 72, alpha=0.1, color='#4574A2FF')
axs.axvspan(135, 155, alpha=0.1, color='grey')

axs.axhline(12.5, color = 'black', ls = "--", alpha = 0.7)
axs.axhline(110, color = 'black', ls = "--", alpha = 0.7)
axs.axvline(72, color = 'black', ls = "-", alpha = 0.7)
axs.axhspan(12.5, 110, color="grey", alpha=0.1)
# axs.annotate('on',(8.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs.axvspan(20,30, alpha=0.1, color='#4574A2FF')
# axs.annotate('on',(23.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs.axvspan(35,45, alpha=0.1, color='#4574A2FF')
# axs.annotate('on',(38.5,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs.axvspan(-4,5, alpha=0.1, color='gray')
# axs.annotate('Phase I', (42, 122), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha="center")
# axs.annotate('Phase II', (103.5, 122), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha="center")
# axs.annotate('88%\ndecrease', (103.5, 61.75), xytext=None, xycoords='data', textcoords='data', fontsize=24, ha="center", va="center")

plt.savefig("ExtData-Fig2-Hires.png", format="png", bbox_inches="tight", dpi=fig.dpi)

-------------
# On/Off electrochemical cell data plotting (SI Figure 14)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
# fig.tight_layout()


axs.errorbar(id478['binTime(m)'], id478['Neutrons(cps)'], id478['delta-neutrons'], ls ='', marker = 'o', ms = 10, capsize = 0, color ='#424242FF' )
axs.tick_params(axis="y", labelsize = 24)
# axs.set_xlim(-4,125)
axs.tick_params(axis="x", labelsize = 24)
axs.set_ylim(-5,220)
axs.xaxis.set_minor_locator(AutoMinorLocator(3))
axs.yaxis.set_minor_locator(AutoMinorLocator(3))
axs.set_ylabel('Neutron production rate (1/s)', fontsize=24)
loc = plticker.MultipleLocator(base=30) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)
axs.axvspan(60,90, alpha=0.1, color='#4574A2FF')
axs.annotate('on',(67,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.axvspan(120,150, alpha=0.1, color='#4574A2FF')

axs.annotate('on',(127,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.axvspan(180,210, alpha=0.1, color='#4574A2FF')
axs.annotate('on',(187,25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs[0].axvspan(-4,5, alpha=0.1, color='gray')
axs.set_xlabel('Time (min)', fontsize = 24)
# axs.annotate('b',(-10,205), xytext = None, xycoords = 'canvas', textcoords = 'data', fontsize = 40 )

axs.annotate('off',(97, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.annotate('off',(158, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
axs.annotate('off',(220, 25), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )

# axs[1].annotate('beam\nloading',(10, 18), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 24 )
# axs.axvline(150)
# axs.axvline(103)
# axs.axhline(175)
# axs.axhline(150)

plt.savefig("SI-Fig13-EcellPowerCycle.png", format="png", bbox_inches="tight")
# for ax in axs.flat:
    # ax.label_outer()
plt.show()




---------

## Determine the saturation parameters
### The loading curves are said to reach saturation when the values are within 5% of the avergae value over 30 min. Below is the calculation for the beam loading curves

### Target A5

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8,6))
fig.tight_layout()

axs.errorbar(id419['binTime(s)'], id419['Neutrons(cps)'], id419['delta-neutrons'], ls ='', marker = 's', ms = 5, capsize = 4, label = 'Beam loading', color ='#424242FF' )
axs.errorbar(id423['binTime(s)'], id423['Neutrons(cps)'], id423['delta-neutrons'], ls ='', marker = '^', ms = 6, capsize = 4, label = 'Beam + e-cell', color ='#941100FF' )
axs.legend(fontsize = 13, loc = 'lower right')
# axs.set_title('Target = A5', fontsize = 16)
axs.tick_params(axis="y", labelsize = 14)
axs.set_xlim(2000,4000)
axs.tick_params(axis="x", labelsize = 14)
axs.set_ylim(125,160)
axs.set_ylabel('Neutron rate [1/s]', fontsize=16)
axs.set_xlabel('Time [s]', fontsize=16)
axs.axhline(135.5, color = 'black', ls = "-", alpha = 0.7)
axs.axhline(142.25, color = 'black', ls = "--", alpha = 0.7)
axs.axhline(128.75, color = 'black', ls = "--", alpha = 0.7)


axs.axvline(2100, color = 'black', ls="-", alpha = 0.7)
axs.axvline(3900, color = 'black', ls="-", alpha = 0.7)

loc = plticker.MultipleLocator(base=1000) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)
axs.axvspan(-250,3900, alpha=0.1, color='#4574A2FF')

# plt.savefig("Plot.pdf", dpi=300, pad_inches = 0.1)

plt.show()

### Calculate the average for the beam loading

In [ ]:
id423satb = id423[(id423['binTime(s)'] > 2100) & (id423['binTime(s)'] < 3900)]
id423satb.head(20)

In [ ]:
aveSatb = sum(id423satb['Neutrons(cps)'])/len(id423satb)
stdDerrSatb = statistics.stdev(id423satb['Neutrons(cps)'])/sqrt(len(id423satb['Neutrons(cps)']))
print(aveSatb, stdDerrSatb)

------------------
### Target A6

In [ ]:
id418satb = id418[(id418['binTime(s)'] > 2400) & (id418['binTime(s)'] < 4200)]
id418satb.head(20)

### Calculate the average for the beam loading

In [ ]:
aveSatb2 = sum(id418satb['Neutrons(cps)'])/len(id418satb)
stdDerrSatb2 = statistics.stdev(id418satb['Neutrons(cps)'])/sqrt(len(id418satb['Neutrons(cps)']))
print(aveSatb2, stdDerrSatb2)

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8,6))
fig.tight_layout()

axs.errorbar(id421['binTime(s)'], id421['Neutrons(cps)'], id421['delta-neutrons'], ls ='', marker = 's', ms = 5, capsize = 4, label = 'Beam loading', color ='#424242FF' )
axs.errorbar(id418['binTime(s)'], id418['Neutrons(cps)'], id418['delta-neutrons'], ls ='', marker = '^', ms = 6, capsize = 4, label = 'Beam + e-cell', color ='#941100FF' )
axs.legend(fontsize = 13, loc = 'lower right')
# axs.set_title('Target = A5', fontsize = 16)
axs.tick_params(axis="y", labelsize = 14)
axs.set_xlim(1800,4250)
axs.tick_params(axis="x", labelsize = 14)
axs.set_ylim(125,160)
axs.set_ylabel('Neutron rate [1/s]', fontsize=16)
axs.set_xlabel('Time [s]', fontsize=16)
axs.axhline(137.2, color = 'black', ls = "-", alpha = 0.7)
axs.axhline(144.08, color = 'black', ls = "--", alpha = 0.7)
axs.axhline(130.36, color = 'black', ls = "--", alpha = 0.7)
axs.axvline(4200, color = 'black', ls = "--", alpha = 0.7)
axs.axvline(2400, color = 'black', ls = "--", alpha = 0.7)
loc = plticker.MultipleLocator(base=1000) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)
axs.axvspan(-250,3900, alpha=0.1, color='#4574A2FF')

# plt.savefig("Plot.pdf", dpi=300, pad_inches = 0.1)

plt.show()

-----------
### Target A7

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (8,6))
fig.tight_layout()

axs.errorbar(id415['binTime(s)'], id415['Neutrons(cps)'], id415['delta-neutrons'], ls ='', marker = 's', ms = 5, capsize = 4, label = 'Beam loading', color ='#424242FF' )
axs.errorbar(id422['binTime(s)'], id422['Neutrons(cps)'], id422['delta-neutrons'], ls ='', marker = '^', ms = 6, capsize = 4, label = 'Beam + e-cell', color ='#941100FF' )
axs.legend(fontsize = 13, loc = 'lower right')
# axs.set_title('Target = A5', fontsize = 16)
axs.tick_params(axis="y", labelsize = 14)
axs.set_xlim(2000,4000)
axs.tick_params(axis="x", labelsize = 14)
axs.set_ylim(125,160)
axs.set_ylabel('Neutron rate [1/s]', fontsize=16)
axs.set_xlabel('Time [s]', fontsize=16)
axs.axhline(136, color = 'black', ls = "-", alpha = 0.7)
axs.axhline(129.23, color = 'black', ls = "--", alpha = 0.7)
axs.axhline(142.8, color = 'black', ls = "--", alpha = 0.7)
axs.axhline(2200, color = 'black', ls = "-", alpha = 0.7)

axs.axvline(3960, color = 'black', ls = "-", alpha = 0.7)
axs.axvline(2160, color = 'black', ls = "-", alpha = 0.7)
loc = plticker.MultipleLocator(base=1000) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)
axs.axvspan(-250,3900, alpha=0.1, color='#4574A2FF')

# plt.savefig("Plot.pdf", dpi=300, pad_inches = 0.1)

plt.show()

In [ ]:
id422satb = id422[(id422['binTime(s)'] > 2160) & (id422['binTime(s)'] < 3960)]
id422satb.head(20)

### Calculate the average for the beam loading

In [ ]:
aveSatb3 = sum(id422satb['Neutrons(cps)'])/len(id422satb)
stdDerrSatb3 = statistics.stdev(id422satb['Neutrons(cps)'])/sqrt(len(id422satb['Neutrons(cps)']))
print(aveSatb3, stdDerrSatb3)

-----------------------

# Beamloading curves
### Plot for Figure 2c in the manuscript

In [ ]:
id444 = readNeutronData(data_folder + 'Fig3a/Data/ID-444_data_300s_bin.csv')
id446 = readNeutronData(data_folder + 'Fig3a/Data/ID-446_data_300s_bin.csv')
id447 = readNeutronData(data_folder + 'Fig3a/Data/ID-447_data_300s_bin.csv')
id449 = readNeutronData(data_folder + 'Fig3a/Data/ID-449_data_300s_bin.csv')
id450 = readNeutronData(data_folder + 'Fig3a/Data/ID-450_data_300s_bin.csv')
id451 = readNeutronData(data_folder + 'Fig3a/Data/ID-451_data_300s_bin.csv')
id454 = readNeutronData(data_folder + 'Fig3a/Data/ID-454_data_300s_bin.csv')

In [ ]:
fig, axs = plt.subplots(1, 1, figsize = (7,7))
fig.tight_layout()

axs.errorbar(id444['binTime(m)'], id444['Neutrons(cps)'], id444['delta-neutrons'], ls ='', marker = 'o', ms = 10, label = 'Beam loaded', color = '#941100FF' )
axs.errorbar(id446['binTime(m)'], id446['Neutrons(cps)'], id446['delta-neutrons'], ls ='', marker = 's', ms = 10, label = 'Beam loaded', color = '#424242FF')
axs.errorbar(id447['binTime(m)'], id447['Neutrons(cps)'], id447['delta-neutrons'], ls ='', marker = 'P', ms = 10, label = 'Beam loaded', color = '#941100FF')
axs.errorbar(id449['binTime(m)'], id449['Neutrons(cps)'], id449['delta-neutrons'], ls ='', marker = 'p', ms = 10, label = 'Beam loaded', color = '#424242FF')
axs.errorbar(id450['binTime(m)'], id450['Neutrons(cps)'], id450['delta-neutrons'], ls ='', marker = 'h', ms = 10, label = 'Beam loaded', color = '#941100FF')
axs.errorbar(id451['binTime(m)'], id451['Neutrons(cps)'], id451['delta-neutrons'], ls ='', marker = 'X', ms = 10, label = 'Beam loaded', color = '#424242FF')
axs.errorbar(id454['binTime(m)'], id454['Neutrons(cps)'], id454['delta-neutrons'], ls ='', marker = '8', ms = 10, label = 'Beam loaded', color = '#941100FF')
# axs[0].legend(fontsize = 14, loc = 'lower right')
# axs[0].set_title('Target A1', fontsize = 32)
axs.tick_params(axis="y", labelsize = 26)
axs.set_xlim(-1,120)
axs.tick_params(axis="x", labelsize = 26)
axs.set_ylim(-1,80)
axs.xaxis.set_minor_locator(AutoMinorLocator(3))
axs.yaxis.set_minor_locator(AutoMinorLocator(3))
axs.set_ylabel('Neutron production rate (1/s)', fontsize=26)
axs.set_xlabel('Time (min)', fontsize=26)
# axs.annotate('E-cell start \n15% increase',(4000,115), xytext = None, xycoords = 'data', textcoords = 'data', fontsize = 14 )
loc = plticker.MultipleLocator(base=20) # this locator puts ticks at regular intervals
axs.xaxis.set_major_locator(loc)


plt.savefig("Fig2a-BeamLoadingCurves2.png", format="png", bbox_inches="tight")
plt.show()

----